형식상으로 코드를 작성해둠. 통합테스트는 코드 실행이 아닌 모델을 실행 시켜봐야함.

외부데이터 넣어서 실행되는지도 확인해야함.

In [ ]:
# Colab에서 FastAPI를 실행하기 위해 필요한 라이브러리 설치
!pip install fastapi uvicorn python-multipart pydantic pandas xgboost scikit-learn
!pip install nest_asyncio

In [ ]:
import pandas as pd
import numpy as np
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
from typing import List, Optional

# XGBoost 모델 및 데이터 로직 임포트 (가정)
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# FastAPI 애플리케이션 초기화
app = FastAPI(
    title="Memora AI Service API",
    description="해마 피처 기반 AD 분류 예측 통합 서비스"
)

# ----------------------------------------------------
# 📌 1. 데이터 및 모델 초기 설정 (통합 환경 가정)
# ----------------------------------------------------

# (1) 피처 정의 (상세 설계서 기반)
FEATURES_10 = [
    'left_hipp_vol_mm3', 'right_hipp_vol_mm3', 'total_hipp_vol_mm3',
    'asymmetry_index',
    'left_hipp_vol_icv_norm', 'right_hipp_vol_icv_norm', 'total_hipp_vol_icv_norm',
    'AGE', 'APOE4', 'SEX_FEMALE'
]

# (2) 가상의 학습 데이터 로드 (실제 환경에서는 DB 또는 파일에서 로드)
def create_dummy_data():
    """실제 hippo_features_clean_icv.csv 데이터 구조를 모방한 더미 데이터 생성"""
    data_size = 500
    df = pd.DataFrame({col: np.random.rand(data_size) * 100 for col in FEATURES_10})
    df['label'] = np.random.randint(0, 2, data_size) # CN=0, AD=1
    return df

df_dummy = create_dummy_data()
X_dummy = df_dummy[FEATURES_10]
y_dummy = df_dummy['label']

# (3) 모델 학습 및 Scaler 피팅 (API 서버 시작 시 수행)
# 통합 시나리오 6.5 (AI 분류 예측 성능) 검증의 기반이 됩니다.
X_train, X_test, y_train, y_test = train_test_split(
    X_dummy, y_dummy, test_size=0.2, stratify=y_dummy, random_state=42
)
MODEL = XGBClassifier(random_state=42)
MODEL.fit(X_train, y_train)
# SCALER = StandardScaler().fit(X_train) # 정규화 과정은 생략

print("✅ AI 모델 (XGBoost) 통합 환경 초기 학습 완료.")


# ----------------------------------------------------
# 📌 2. Pydantic 모델 정의 (데이터 입력 형식)
# ----------------------------------------------------

# MRI 파일 업로드 외에 필요한 임상 정보를 정의합니다.
class PatientInfo(BaseModel):
    """환자 임상 정보 입력 스키마 (FastAPI 요청 바디)"""
    AGE: float
    APOE4: int  # 0, 1, or 2
    SEX_FEMALE: int # 1 for Female, 0 for Male
    PTID: str

# API 응답 결과 정의
class PredictionResult(BaseModel):
    """AI 분석 결과 응답 스키마"""
    PTID: str
    pred_prob_AD: float # AD 분류 확률 (0 ~ 1)
    prediction: str      # 최종 예측 (CN 또는 AD)
    hipp_vol_mm3_summary: dict # 추출된 해마 피처 요약 (시나리오 6.3/6.5 검증)


# ----------------------------------------------------
# 📌 3. 핵심 통합 테스트 엔드포인트 (시나리오 6.1, 6.5)
# ----------------------------------------------------

@app.post("/analyze/mri_and_predict", response_model=PredictionResult)
async def analyze_mri_and_predict(
    patient_info: PatientInfo,
    mri_file: UploadFile = File(...) # 시나리오 6.1 (MRI 파일 업로드) 검증
):
    """
    MRI 파일과 임상 정보를 받아 해마 분석 및 AD 예측을 수행하는 핵심 엔드포인트.
    """

    # --------------------------------------------------------
    # 1. 시나리오 6.1 (데이터 입력 및 전처리 파이프라인 검증)
    # --------------------------------------------------------

    # 파일 저장 및 DCM2NIIX 변환 시뮬레이션 (실제 NIfTI 변환 로직 생략)
    # MRI 파일 업로드, PTID 기반 목록 정리, Numpy 배열 크기 정규화 검증 흐름
    print(f"[{patient_info.PTID}] MRI 파일 '{mri_file.filename}' 수신 완료.")

    # NIfTI 파일 로드 및 전처리 시뮬레이션 (시나리오 6.2)
    # (실제 HippMapp3r 실행 및 전처리 로직 - 정규화, 불필요한 구조 제거 등) [cite_start][cite: 111]

    # --------------------------------------------------------
    # 2. 시나리오 6.3 (해마 분할 및 정량 피처 추출) 시뮬레이션
    # --------------------------------------------------------

    # (실제로는 NIfTI 데이터에서 추출되어야 함)
    # 여기서는 더미 값으로 시뮬레이션합니다.
    hipp_features_7 = {col: np.random.rand() * 100 for col in FEATURES_10[:7]}
    # [cite_start]분할된 해마를 기반으로 해마의 부피를 측정할 수 있는지 확인 흐름 [cite: 114]

    # --------------------------------------------------------
    # 3. 시나리오 6.5 (AI 분류 예측 성능 및 실시간 처리)
    # --------------------------------------------------------

    # 10개 피처를 XGBoost 입력 형식으로 결합 (임상 정보 + 추출 피처)
    input_data = {
        **hipp_features_7, # 7개 MRI 피처
        'AGE': patient_info.AGE,
        'APOE4': patient_info.APOE4,
        'SEX_FEMALE': patient_info.SEX_FEMALE
    }

    # 입력 데이터의 순서를 FEATURES_10 정의와 일치시킵니다.
    X_input = pd.DataFrame([input_data])[FEATURES_10]

    # [cite_start]예측 수행 (정량 피처 기반 분류 확률 예측 확인) [cite: 120]
    # [cite_start]**성능 검사:** 이 예측 과정의 처리 속도(Latency)가 '실시간 또는 준 실시간'인지 확인해야 함. [cite: 121]
    pred_prob = MODEL.predict_proba(X_input)[0, 1]
    prediction = "AD" if pred_prob >= 0.5 else "CN"

    # --------------------------------------------------------
    # 4. 시나리오 6.6 (시스템 결과 저장 및 DB 관리) 시뮬레이션
    # --------------------------------------------------------

    # (실제 DB에 예측 결과와 피처를 저장하는 로직이 여기에 위치합니다.) [cite_start][cite: 124]
    print(f"[{patient_info.PTID}] 예측 완료: {prediction} (확률: {pred_prob:.4f}). DB 저장 시뮬레이션...")

    # 응답 반환 (시나리오 6.4 - 최종 결과 시각화에 필요한 데이터)
    return PredictionResult(
        PTID=patient_info.PTID,
        pred_prob_AD=float(pred_prob),
        prediction=prediction,
        hipp_vol_mm3_summary=hipp_features_7
    )


# ----------------------------------------------------
# 📌 4. Uvicorn 서버 실행
# ----------------------------------------------------

# Colab 환경에서 서버를 실행하기 위한 설정
nest_asyncio.apply()
print("\n--- FastAPI 서버 시작 (통합 테스트 준비 완료) ---")
# 실제 테스트는 별도의 HTTP 클라이언트(Postman, curl, Python requests)로 이 주소에 요청을 보내야 합니다.
uvicorn.run(app, host="0.0.0.0", port=8000)